In [1]:
from __future__ import annotations

import argparse
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
# print(xgb.__version__)

In [2]:
DEFAULT_CSV = "cleaned_secondary_final.csv"
ARTIFACT_NAME = "xgboost_secondary_model.joblib"

In [3]:
FEATURE_COLUMNS = [
    "ret_stock",
    "ret_dos",
    "wod",
    "activating_outlet",
    "dbr_stock",
    "dbr_dos",
    "stocking_outlet",
    "tertiary",
    "seasonality_1(sin)",
    "seasonality_2(cos)",
    "secondary_lag_1",
    "secondary_lag_2",
    "secondary_lag_4",
    "secondary_lag_8",
    "secondary_roll_mean_4",
    "secondary_roll_mean_8",
    "secondary_growth_1",
]

LOG_FEATURE_COLS = [
    "ret_stock",
    "ret_dos",
    "wod",
    "activating_outlet",
    "dbr_stock",
    "dbr_dos",
    "stocking_outlet",
    "tertiary",
]

TARGET_COLUMN = "secondary"

In [4]:
def calculate_metrics(y_true, y_pred, model_name="Model"):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    denominator = np.clip(np.abs(y_true), 1, None)
    mape = np.mean(np.abs((y_true - y_pred) / denominator)) * 100
    accuracy = max(0.0, 100 - mape)

    return {"R2": r2, "MAE": mae, "RMSE": rmse, "MAPE": mape, "Accuracy": accuracy}

In [5]:
def _fmt_mape_gt10(m: dict) -> str:
    v = m.get("MAPE_gt10", float("nan"))
    return f"{v:.2f}%" if not np.isnan(v) else "n/a"

def _fmt_acc_gt10(m: dict) -> str:
    v = m.get("Accuracy_gt10", float("nan"))
    return f"{v:.2f}%" if not np.isnan(v) else "n/a"


def add_seasonality(df: pd.DataFrame) -> pd.DataFrame:
    """Match inference: calendar week index in df['week'], period 59."""
    out = df.copy()
    if "week" not in out.columns:
        return out
    _w = pd.to_numeric(out["week"], errors="coerce").fillna(0)
    _ang = 2.0 * np.pi * (_w / 59.0)
    out["seasonality_1(sin)"] = np.sin(_ang)
    out["seasonality_2(cos)"] = np.cos(_ang)
    return out

In [6]:
def add_lag_features(df):
    df = df.copy()

    df = df.sort_values(["model_family", "year", "week"])

    g = df.groupby(["model_family"])["secondary"]

    for lag in (1, 2, 4, 8):
        df[f"secondary_lag_{lag}"] = g.shift(lag)

    for w in (4, 8):
        df[f"secondary_roll_mean_{w}"] = g.transform(
            lambda s: s.shift(1).rolling(window=w, min_periods=w).mean()
        )

    df["secondary_growth_1"] = df["secondary"] / (df["secondary_lag_1"] + 1e-5)

    return df


In [7]:
def load_and_prepare(csv_path):

    df = pd.read_csv(csv_path)

    df["start_date"] = pd.to_datetime(df["start_date"])
    df["year"] = df["start_date"].dt.year

    # minimal cleaning
    df = df[(df["secondary"] >= 0)].copy()

    # CAP → save for inference
    cap = df["secondary"].quantile(0.955)
    df = df[df["secondary"] <= cap]

    # fill missing
    for col in df.select_dtypes(include=[np.number]).columns:
        df[col] = df[col].fillna(df[col].median())

    df = df.drop_duplicates()

    # correct ordering
    df = df.sort_values(["model_family", "year", "week"]).reset_index(drop=True)

    # feature engineering
    df = add_lag_features(df)

    lag_cols = [c for c in df.columns if "lag_" in c or "roll_mean" in c]
    df = df.dropna(subset=lag_cols)

    df = add_seasonality(df)

    return df, cap


In [8]:
def split_train_test(df):

    # global time-aware split (correct way)
    df = df.sort_values(["year", "week"]).reset_index(drop=True)

    split_idx = int(len(df) * 0.8)

    df_train = df.iloc[:split_idx]
    df_test = df.iloc[split_idx:]

    return df_train, df_test

In [9]:
def apply_log(X, cols):
    X = X.copy()
    for c in cols:
        if c in X.columns:
            X[c] = np.log1p(np.clip(X[c], 0, None))
    return X

In [10]:
def align_features(
    df_train: pd.DataFrame,
    df_test: pd.DataFrame,
    feature_cols: list[str],
) -> tuple[list[str], pd.DataFrame, pd.DataFrame]:
    cols = list(feature_cols)
    missing = [c for c in cols if c not in df_train.columns]
    if missing:
        print(f"Warning: dropping missing columns: {missing}")
        cols = [c for c in cols if c in df_train.columns]
    for c in cols:
        if c not in df_train.columns:
            df_train[c] = 0.0
        if c not in df_test.columns:
            df_test[c] = 0.0
    return cols, df_train, df_test


def apply_log_features(X: pd.DataFrame, log_cols: list[str]) -> pd.DataFrame:
    out = X.copy()
    for c in log_cols:
        if c in out.columns:
            v = pd.to_numeric(out[c], errors="coerce").astype(float)
            v = v.clip(lower=0)
            out[c] = np.log1p(v)
    return out

In [11]:
def build_xy(df_train, df_test):

    X_train = df_train[FEATURE_COLUMNS].fillna(0)
    X_test = df_test[FEATURE_COLUMNS].fillna(0)

    y_train = df_train[TARGET_COLUMN].values
    y_test = df_test[TARGET_COLUMN].values

    X_train = apply_log(X_train, LOG_FEATURE_COLS)
    X_test = apply_log(X_test, LOG_FEATURE_COLS)

    return X_train, X_test, np.log1p(y_train), y_test

In [12]:
def train_model(X_train, y_train):
    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=600,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.85,
        random_state=42,
        n_jobs=-1,
        tree_method="hist",
    )
    model.fit(X_train, y_train)
    return model

In [13]:
import os
def main():

    csv_path = Path(DEFAULT_CSV)

    df, cap = load_and_prepare(csv_path)

    df_train, df_test = split_train_test(df)

    X_train, X_test, y_train_log, y_test = build_xy(df_train, df_test)

    model = train_model(X_train, y_train_log)

    pred = np.expm1(model.predict(X_test))
    pred = np.clip(pred, 0, None)

    metrics = calculate_metrics(y_test, pred)

    print("\n--- TEST METRICS ---")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")

    # SAVE ARTIFACT
    artifact = {
        "model": model,
        "features": FEATURE_COLUMNS,
        "log_feature_cols": LOG_FEATURE_COLS,
        "target": TARGET_COLUMN,
        "target_log1p": True,
        "target_cap": cap,   # 🔥 IMPORTANT
    }

    joblib.dump(artifact, ARTIFACT_NAME)
    print(f"Saved: {ARTIFACT_NAME}")




In [14]:
if __name__ == "__main__":
    main()


--- TEST METRICS ---
R2: 0.9526
MAE: 91.0074
RMSE: 368.9501
MAPE: 9.3723
Accuracy: 90.6277
Saved: xgboost_secondary_model.joblib
